# Preprocessing — Eksperimen 4

**Alur:** load data -> split -> delexicalization -> belief span -> format sample per turn -> tokenisasi -> vocabulary -> word ke index -> padding -> tensor -> knowledge base query.

**Optimasi vs eksp3:**
1. **Delexicalization word-boundary** — semua slot (termasuk nama/alamat/telepon/kodepos) dicocokkan dengan `\b...\b`, bukan substring mentah, agar nilai tidak mencemari kata lain (mis. `asked` tidak berubah jadi `NAME_SLOTed`).
2. **Vocab hygiene (`min_freq=2`)** — token yang hanya muncul sekali (hapax, banyak berupa typo) dibuang dari vocabulary sehingga menjadi OOV. Ini **memaksa mekanisme copy TSCP terlatih** untuk menyalin nilai langka (mis. `cuban`, `seafood`) langsung dari input — sesuai keunggulan OOV pada paper.

## 1. Konfigurasi

Definisi rasio split data, token khusus (`<pad>`, `<sos>`, `<eos>`, `<unk>`, `<inf>`, `<req>`, slot token), dan daftar slot informable.

In [2]:
import json, re, random
from pathlib import Path
from collections import Counter

import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"

# Split 3:1:1 sesuai paper
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.6, 0.2, 0.2
SEED = 42

# Special tokens
PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN = "<pad>", "<sos>", "<eos>", "<unk>"
INF_OPEN, INF_CLOSE, REQ_OPEN, REQ_CLOSE = "<Inf>", "</Inf>", "<Req>", "</Req>"
SLOT_TOKENS = ["NAME_SLOT", "ADDRESS_SLOT", "PHONE_SLOT", "POSTCODE_SLOT",
               "FOOD_SLOT", "AREA_SLOT", "PRICERANGE_SLOT"]
SPECIAL_TOKENS = [PAD_TOKEN, SOS_TOKEN, EOS_TOKEN, UNK_TOKEN,
                  INF_OPEN, INF_CLOSE, REQ_OPEN, REQ_CLOSE] + SLOT_TOKENS

# Slot CamRest676
INFORMABLE_SLOTS = ["food", "area", "pricerange"]
DB_FIELD_TO_SLOT = {"name": "NAME_SLOT", "address": "ADDRESS_SLOT", "phone": "PHONE_SLOT",
                    "postcode": "POSTCODE_SLOT", "food": "FOOD_SLOT", "area": "AREA_SLOT",
                    "pricerange": "PRICERANGE_SLOT"}

print("Root repo:", ROOT)
print("Data dir :", DATA_DIR)
print(f"Special tokens ({len(SPECIAL_TOKENS)}):", SPECIAL_TOKENS)

Root repo: e:\coding bebas\restorant-asistent
Data dir : e:\coding bebas\restorant-asistent\data
Special tokens (15): ['<pad>', '<sos>', '<eos>', '<unk>', '<Inf>', '</Inf>', '<Req>', '</Req>', 'NAME_SLOT', 'ADDRESS_SLOT', 'PHONE_SLOT', 'POSTCODE_SLOT', 'FOOD_SLOT', 'AREA_SLOT', 'PRICERANGE_SLOT']


## 2. Load Data Mentah

Baca 2 file: `CamRest676.json` (676 dialog percakapan restoran) dan `CamRestDB.json` (database restoran). Kedua file ini jadi sumber untuk semua tahap berikutnya.

In [3]:
def load_raw_data():
    with open(DATA_DIR / "CamRest676.json") as f:
        dialogues = json.load(f)
    with open(DATA_DIR / "CamRestDB.json") as f:
        database = json.load(f)
    return dialogues, database

dialogues, database = load_raw_data()

turn = dialogues[0]["dial"][0]
print(f"Total dialog : {len(dialogues)}")
print(f"Total KB     : {len(database)} restoran\n")
print("Contoh 1 turn mentah:")
print("  usr:", turn["usr"]["transcript"])
print("  sys:", turn["sys"]["sent"])
print("\nContoh entry KB:")
print(" ", database[0])

Total dialog : 676
Total KB     : 110 restoran

Contoh 1 turn mentah:
  usr: I need to find an expensive restauant that's in the south section of the city.
  sys: There are several restaurants in the south part of town that serve expensive food. Do you have a cuisine preference?

Contoh entry KB:
  {'address': 'Regent Street City Centre', 'area': 'centre', 'food': 'italian', 'location': '52.20103,0.126023', 'phone': '01223 323737', 'pricerange': 'cheap', 'postcode': 'C.B 2, 1 A.B', 'type': 'restaurant', 'id': '19210', 'name': 'pizza hut city centre'}


## 3. Split Data (3:1:1)

Bagi dialog jadi train/val/test dengan rasio 3:1:1, mengikuti setting asli paper. Shuffle pakai `seed` tetap supaya hasil split selalu sama tiap dijalankan ulang (reproducible).

In [4]:
def split_data(dialogues, seed=SEED):
    random.seed(seed)
    idx = list(range(len(dialogues)))
    random.shuffle(idx)
    n = len(dialogues)
    n_train, n_val = int(n * TRAIN_RATIO), int(n * VAL_RATIO)
    train = [dialogues[i] for i in idx[:n_train]]
    val = [dialogues[i] for i in idx[n_train:n_train + n_val]]
    test = [dialogues[i] for i in idx[n_train + n_val:]]
    return train, val, test

train_dial, val_dial, test_dial = split_data(dialogues)
print(f"Train: {len(train_dial)} | Val: {len(val_dial)} | Test: {len(test_dial)} dialog")

Train: 405 | Val: 135 | Test: 136 dialog


## 4. Delexicalization

Ganti nilai spesifik restoran (nama, alamat, telepon, dll) di **response sistem** dengan placeholder generik (`NAME_SLOT`, `ADDRESS_SLOT`, ...).

**Tujuan:** model tidak perlu menghafal ratusan nama/alamat restoran, cukup belajar pola kalimat. Nilai asli ditempel kembali saat inference lewat proses lexicalization.

Hanya response yang di-delex; input user dan belief span dibiarkan apa adanya.

**Catatan (optimasi eksp4):** semua slot dicocokkan dengan **word-boundary** (`...`) — bukan substring mentah seperti eksp3 — supaya nilai tidak mencemari kata lain (mis. restoran bernama `ask` tidak lagi mengubah `asked` menjadi `NAME_SLOTed`). Nilai diurutkan dari yang terpanjang agar frasa panjang menang atas frasa pendek.

In [5]:
def collect_slot_values(database):
    """Kumpulkan pasangan (value, slot) unik dari DB, diurutkan value terpanjang dulu.

    Urutan terpanjang-dulu mencegah frasa pendek menimpa frasa panjang
    (mis. 'city centre' harus diproses sebelum 'centre').
    """
    pairs, seen = [], set()
    for entry in database:
        for field, slot in DB_FIELD_TO_SLOT.items():
            val = str(entry.get(field, "")).lower().strip()
            if val and (val, slot) not in seen:
                seen.add((val, slot))
                pairs.append((val, slot))
    pairs.sort(key=lambda p: len(p[0]), reverse=True)
    return pairs


_WB = chr(92) + "b"  # penanda word-boundary regex, ditulis via chr(92) agar aman


def delexicalize_response(response, slot_value_pairs):
    """Ganti nilai entitas di response dengan placeholder slot.

    Semua slot memakai word-boundary agar nilai tidak mencemari kata lain
    (mis. 'asked' tidak menjadi 'NAME_SLOTed').
    """
    delex = response.lower()
    for val, slot in slot_value_pairs:
        delex = re.sub(_WB + re.escape(val) + _WB, slot, delex)
    return delex


SLOT_VALUE_PAIRS = collect_slot_values(database)

# Demo: word-boundary tidak merusak kata biasa yang memuat substring nama restoran.
demo_text = "here is the information you asked for."
print("SEBELUM:", demo_text)
print("SESUDAH:", delexicalize_response(demo_text, SLOT_VALUE_PAIRS), "(kata asked tetap utuh)")

sample = next(t["sys"]["sent"] for d in train_dial for t in d["dial"]
              if delexicalize_response(t["sys"]["sent"], SLOT_VALUE_PAIRS) != t["sys"]["sent"].lower())
print()
print("SEBELUM:", sample)
print("SESUDAH:", delexicalize_response(sample, SLOT_VALUE_PAIRS))

SEBELUM: here is the information you asked for.
SESUDAH: here is the information you asked for. (kata asked tetap utuh)

SEBELUM: Pizza Hut Cherry Hinton is in the south part of town and in the moderate price range.
SESUDAH: NAME_SLOT is in the AREA_SLOT part of town and in the PRICERANGE_SLOT price range.


## 5. Belief Span (bspan)

Ringkas kebutuhan user per turn jadi satu teks berformat `<inf> value ; value </inf> <req> slot ; slot </req>`.

- **Informable** (`<inf>`): nilai yang user sebutkan sebagai kriteria pencarian (misal `italian`).
- **Requestable** (`<req>`): nama slot yang user minta (misal `address`).

**Tujuan:** menggantikan peran intent + slot classifier lewat satu representasi teks yang bisa di-generate model.

In [6]:
def construct_bspan(slu_annotations):
    informable, requestable = [], []
    for slu in slu_annotations:
        act = slu["act"]
        for pair in slu["slots"]:
            if act == "inform":
                name, value = pair[0], pair[1]
                if value != "dontcare" and name in INFORMABLE_SLOTS and value.lower() not in informable:
                    informable.append(value.lower())
            elif act == "request":
                req = pair[1] if pair[0] == "slot" else pair[0]
                if req.lower() not in requestable:
                    requestable.append(req.lower())
    return (f"{INF_OPEN} {' ; '.join(informable)} {INF_CLOSE} "
            f"{REQ_OPEN} {' ; '.join(requestable)} {REQ_CLOSE}")

# Ambil turn yang punya act request agar informable & requestable sama-sama terlihat.
demo = next(t for d in train_dial for t in d["dial"]
            if any(s["act"] == "request" for s in t["usr"]["slu"]))
print("SLU  :", demo["usr"]["slu"])
print("Bspan:", construct_bspan(demo["usr"]["slu"]))

SLU  : [{'act': 'request', 'slots': [['slot', 'phone']]}, {'act': 'request', 'slots': [['slot', 'food']]}, {'act': 'request', 'slots': [['slot', 'address']]}, {'act': 'inform', 'slots': [['pricerange', 'moderate']]}, {'act': 'inform', 'slots': [['area', 'south']]}]
Bspan: <Inf> moderate ; south </Inf> <Req> phone ; food ; address </Req>


## 6. Format Sample per Turn

Susun tiap turn jadi 3 bagian sesuai rumus Sequicity (`B_t = seq2seq(B_{t-1} R_{t-1} U_t)`):

- **input**: gabungan bspan + response turn sebelumnya + ucapan user saat ini
- **target_bspan**: bspan yang harus diprediksi (decoder tahap 1)
- **target_response**: response yang harus diprediksi (decoder tahap 2)

In [7]:
def process_dialogue(dialogue, slot_value_pairs):
    processed, prev_bspan, prev_response = [], "", ""
    for t, turn in enumerate(dialogue["dial"]):
        user = turn["usr"]["transcript"].lower().strip()
        bspan = construct_bspan(turn["usr"]["slu"])
        response = delexicalize_response(turn["sys"]["sent"].lower().strip(), slot_value_pairs)
        parts = [p for p in (prev_bspan, prev_response) if p] + [user]
        processed.append({
            "input": " ".join(parts),
            "target_bspan": bspan,
            "target_response": response,
            "dialogue_id": dialogue.get("dialogue_id", ""),
            "turn": t,
        })
        prev_bspan, prev_response = bspan, response
    return processed

def process_all(dialogues, slot_value_pairs):
    return [s for d in dialogues for s in process_dialogue(d, slot_value_pairs)]

train_samples = process_all(train_dial, SLOT_VALUE_PAIRS)
val_samples = process_all(val_dial, SLOT_VALUE_PAIRS)
test_samples = process_all(test_dial, SLOT_VALUE_PAIRS)

s = train_samples[3]

print("Contoh hasil sample ke-3 (train):")
print("INPUT    :", s["input"])
print("BSPAN    :", s["target_bspan"])
print("RESPONSE :", s["target_response"])
print()
print(f"Total sample -> Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}")

Contoh hasil sample ke-3 (train):
INPUT    : <Inf> moderate ; south </Inf> <Req> phone ; food ; address </Req> they serve FOOD_SLOT food and are located at ADDRESS_SLOT.  their number is PHONE_SLOT. thank you
BSPAN    : <Inf> moderate ; south </Inf> <Req>  </Req>
RESPONSE : you are very welcome. good bye.

Total sample -> Train: 1635, Val: 553, Test: 556


## 7. Tokenisasi

Pecah teks jadi token per kata, pisahkan tanda baca dari kata. Token khusus (`<inf>`, `NAME_SLOT`, dll) dilindungi supaya tidak ikut terpecah oleh proses split ini.

In [8]:
def tokenize(text, protected_tokens):
    sorted_protected = sorted(protected_tokens, key=len, reverse=True)
    pattern_protected = "|".join(re.escape(tok) for tok in sorted_protected)

    segments = re.split(f"({pattern_protected})", text)

    result_tokens = []
    for segment in segments:
        if segment in protected_tokens:
            # Token spesial tidak di tokenisasi
            result_tokens.append(segment)
        elif segment.strip():
            # Teks biasa di tokenisasi
            cleaned = re.sub(r"([.,!?;:'\"\(\)])", r" \1 ", segment)
            cleaned = re.sub(r"\s+", " ", cleaned).strip()
            result_tokens.extend(cleaned.split())

    return result_tokens


def build_tokenized_record(sample):
    tokens_bspan_core = tokenize(sample["target_bspan"], SPECIAL_TOKENS)
    tokens_response_core = tokenize(sample["target_response"], SPECIAL_TOKENS)

    return {
        "tokens_input": tokenize(sample["input"], SPECIAL_TOKENS),
        "tokens_bspan": [SOS_TOKEN] + tokens_bspan_core + [EOS_TOKEN],
        "tokens_response": [SOS_TOKEN] + tokens_response_core + [EOS_TOKEN],
        "dialogue_id": sample["dialogue_id"],
        "turn": sample["turn"],
    }


def tokenize_dataset(samples):
    return [build_tokenized_record(s) for s in samples]


train_tokenized = tokenize_dataset(train_samples)
val_tokenized = tokenize_dataset(val_samples)
test_tokenized = tokenize_dataset(test_samples)

sample_check = train_tokenized[3]

print("\nContoh hasil sample ke-3 (train):")
print("\nTokens input    :", sample_check["tokens_input"])
print("Tokens bspan    :", sample_check["tokens_bspan"])
print("Tokens response :", sample_check["tokens_response"])
print()
print("Total tokenized -> Train:", len(train_tokenized),
      "Val:", len(val_tokenized),
      "Test:", len(test_tokenized))


Contoh hasil sample ke-3 (train):

Tokens input    : ['<Inf>', 'moderate', ';', 'south', '</Inf>', '<Req>', 'phone', ';', 'food', ';', 'address', '</Req>', 'they', 'serve', 'FOOD_SLOT', 'food', 'and', 'are', 'located', 'at', 'ADDRESS_SLOT', '.', 'their', 'number', 'is', 'PHONE_SLOT', '.', 'thank', 'you']
Tokens bspan    : ['<sos>', '<Inf>', 'moderate', ';', 'south', '</Inf>', '<Req>', '</Req>', '<eos>']
Tokens response : ['<sos>', 'you', 'are', 'very', 'welcome', '.', 'good', 'bye', '.', '<eos>']

Total tokenized -> Train: 1635 Val: 553 Test: 556


## 8. Vocabulary

Bangun kamus kata -> index, **hanya dari data training** (supaya tidak ada data leakage dari val/test). Token khusus menempati index awal, sisanya kata diurutkan berdasarkan frekuensi.

**Optimasi eksp4 — `min_freq=2`:** kata yang hanya muncul **sekali** di training (hapax) dibuang dari vocabulary. Sebagian besar hapax adalah typo (`resteraunt`, `epensive`, ...) yang hanya menambah noise. Dengan membuangnya: vocabulary lebih bersih, dan kata langka menjadi **OOV** sehingga model **dipaksa belajar meng-copy** nilai seperti `cuban`/`seafood` langsung dari input (keunggulan OOV pada paper).

In [9]:
def build_vocabulary(tokenized_samples, min_freq=2, max_vocab_size=800):
    """Bangun word2idx dari training set.

    - Token khusus (special tokens) selalu masuk, menempati index awal.
    - Kata dengan frekuensi < min_freq dibuang (jadi OOV) untuk melatih copy.
    - Sisanya diurutkan frekuensi menurun, dipotong pada max_vocab_size.
    """
    counter = Counter()
    for s in tokenized_samples:
        counter.update(s["tokens_input"])
        counter.update(s["tokens_bspan"])
        counter.update(s["tokens_response"])

    word2idx = {tok: i for i, tok in enumerate(SPECIAL_TOKENS)}
    sorted_words = sorted(counter.items(), key=lambda x: (-x[1], x[0]))

    n_dropped_rare = 0
    for word, freq in sorted_words:
        if word in word2idx:
            continue
        if freq < min_freq:
            n_dropped_rare += 1
            continue
        if len(word2idx) >= max_vocab_size:
            break
        word2idx[word] = len(word2idx)

    idx2word = {i: w for w, i in word2idx.items()}

    print(f"Vocab size: {len(word2idx)} (min_freq={min_freq}, max={max_vocab_size})")
    print(f"Token langka dibuang jadi OOV: {n_dropped_rare}")
    print("12 entri pertama:", list(word2idx.items())[:12])
    return word2idx, idx2word


word2idx, idx2word = build_vocabulary(train_tokenized, min_freq=2, max_vocab_size=800)

Vocab size: 613 (min_freq=2, max=800)
Token langka dibuang jadi OOV: 137
12 entri pertama: [('<pad>', 0), ('<sos>', 1), ('<eos>', 2), ('<unk>', 3), ('<Inf>', 4), ('</Inf>', 5), ('<Req>', 6), ('</Req>', 7), ('NAME_SLOT', 8), ('ADDRESS_SLOT', 9), ('PHONE_SLOT', 10), ('POSTCODE_SLOT', 11)]


## 9. Word ke Index

Ubah semua token (kata) jadi angka index sesuai vocabulary. Kata yang tidak ada di vocabulary otomatis diganti `<unk>`.

In [10]:
def tokens_to_indices_copynet(tokens, word2idx):
    unk_idx = word2idx[UNK_TOKEN]
    vocab_size = len(word2idx)
    
    indices = []
    oov_words = []
    oov_map = {} # Memetakan kata OOV ke index sementara (vocab_size, vocab_size+1, dst)
    
    for t in tokens:
        if t in word2idx:
            indices.append(word2idx[t])
        else:
            # Jika kata tidak ada di vocab, beri index khusus untuk mekanisme CopyNet
            if t not in oov_map:
                oov_map[t] = vocab_size + len(oov_words)
                oov_words.append(t)
            indices.append(oov_map[t])
            
    return indices, oov_words

def build_indexed_record(tokenized_sample, word2idx):
    input_indices, input_oov = tokens_to_indices_copynet(tokenized_sample["tokens_input"], word2idx)
    bspan_indices, _ = tokens_to_indices_copynet(tokenized_sample["tokens_bspan"], word2idx)
    response_indices, _ = tokens_to_indices_copynet(tokenized_sample["tokens_response"], word2idx)
    
    return {
        "input_indices": input_indices,
        "input_oov": input_oov, # Menyimpan list kata OOV asli dari input
        "bspan_indices": bspan_indices,
        "response_indices": response_indices,
        "dialogue_id": tokenized_sample["dialogue_id"],
        "turn": tokenized_sample["turn"],
    }

def indexize_dataset(tokenized_samples, word2idx):
    return [build_indexed_record(s, word2idx) for s in tokenized_samples]

train_indexed = indexize_dataset(train_tokenized, word2idx)
val_indexed = indexize_dataset(val_tokenized, word2idx)
test_indexed = indexize_dataset(test_tokenized, word2idx)

print("\nContoh hasil sample ke-3 (train):")
print("  input_indices   :", train_indexed[3]["input_indices"])
print("  input_oov (kata asli yang OOV):", train_indexed[3]["input_oov"])
print("  bspan_indices   :", train_indexed[3]["bspan_indices"])
print("  response_indices:", train_indexed[3]["response_indices"])


Contoh hasil sample ke-3 (train):
  input_indices   : [4, 35, 16, 46, 5, 6, 22, 16, 28, 16, 27, 7, 95, 99, 12, 28, 25, 36, 55, 60, 9, 15, 48, 30, 18, 10, 15, 39, 19]
  input_oov (kata asli yang OOV): []
  bspan_indices   : [1, 4, 35, 16, 46, 5, 6, 7, 2]
  response_indices: [1, 19, 36, 175, 100, 15, 70, 76, 15, 2]


## 10. Padding

Samakan panjang semua sequence dalam satu dataset dengan menambah `<pad>` di bagian yang kurang panjang.

Sekalian dilakukan **shifting** untuk target decoder: `bspan_input`/`response_input` (tanpa token akhir) dan `bspan_target`/`response_target` (tanpa token awal), teknik teacher forcing supaya decoder belajar memprediksi token berikutnya.

In [11]:
def pad_sequences(list_of_indices, pad_idx):
    max_len = max(len(seq) for seq in list_of_indices)
    padded = [seq + [pad_idx] * (max_len - len(seq)) for seq in list_of_indices]
    return padded, max_len


def pad_dataset(indexed_samples, pad_idx):
    input_padded, input_maxlen = pad_sequences(
        [s["input_indices"] for s in indexed_samples], pad_idx
    )
    bspan_input_padded, bspan_in_maxlen = pad_sequences(
        [s["bspan_indices"][:-1] for s in indexed_samples], pad_idx
    )
    bspan_target_padded, bspan_tgt_maxlen = pad_sequences(
        [s["bspan_indices"][1:] for s in indexed_samples], pad_idx
    )
    response_input_padded, resp_in_maxlen = pad_sequences(
        [s["response_indices"][:-1] for s in indexed_samples], pad_idx
    )
    response_target_padded, resp_tgt_maxlen = pad_sequences(
        [s["response_indices"][1:] for s in indexed_samples], pad_idx
    )

    return {
        "input_padded": input_padded,
        "bspan_input_padded": bspan_input_padded,
        "bspan_target_padded": bspan_target_padded,
        "response_input_padded": response_input_padded,
        "response_target_padded": response_target_padded,
        "max_lengths": {
            "input": input_maxlen,
            "bspan_input": bspan_in_maxlen,
            "bspan_target": bspan_tgt_maxlen,
            "response_input": resp_in_maxlen,
            "response_target": resp_tgt_maxlen,
        },
    }


train_padded = pad_dataset(train_indexed, word2idx[PAD_TOKEN])
val_padded = pad_dataset(val_indexed, word2idx[PAD_TOKEN])
test_padded = pad_dataset(test_indexed, word2idx[PAD_TOKEN])

print("\nContoh hasil sample ke-3 (train):")
print("Input padded   :", train_padded["input_padded"][3])
print("Bspan input    :", train_padded["bspan_input_padded"][3])
print("Bspan target   :", train_padded["bspan_target_padded"][3])
print("Response input :", train_padded["response_input_padded"][3])
print("Response target:", train_padded["response_target_padded"][3])

print()
print("Train -> total sample:", len(train_padded["input_padded"]),
      "| max_lengths:", train_padded["max_lengths"])
print("Val   -> total sample:", len(val_padded["input_padded"]),
      "| max_lengths:", val_padded["max_lengths"])
print("Test  -> total sample:", len(test_padded["input_padded"]),
      "| max_lengths:", test_padded["max_lengths"])


Contoh hasil sample ke-3 (train):
Input padded   : [4, 35, 16, 46, 5, 6, 22, 16, 28, 16, 27, 7, 95, 99, 12, 28, 25, 36, 55, 60, 9, 15, 48, 30, 18, 10, 15, 39, 19, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Bspan input    : [1, 4, 35, 16, 46, 5, 6, 7, 0, 0, 0, 0, 0, 0, 0]
Bspan target   : [4, 35, 16, 46, 5, 6, 7, 2, 0, 0, 0, 0, 0, 0, 0]
Response input : [1, 19, 36, 175, 100, 15, 70, 76, 15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Response target: [19, 36, 175, 100, 15, 70, 76, 15, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Train -> total sample: 1635 | max_lengths: {'input': 70, 'bspan_input': 15, 'bspan_target': 15, 'response_input': 50, 'response_target': 50}
Val   -> total sample: 553 | max_lengths: {'input': 61, 'bspan_input': 15, 'bspan_t

## 11. Konversi ke Tensor

Ubah data yang sudah di-padding dari list Python biasa menjadi `torch.Tensor`, format yang bisa diproses PyTorch.

In [12]:
def to_tensor_dataset(padded_dict):
    return {
        "input_tensor": torch.tensor(padded_dict["input_padded"], dtype=torch.long),
        "bspan_input_tensor": torch.tensor(padded_dict["bspan_input_padded"], dtype=torch.long),
        "bspan_target_tensor": torch.tensor(padded_dict["bspan_target_padded"], dtype=torch.long),
        "response_input_tensor": torch.tensor(padded_dict["response_input_padded"], dtype=torch.long),
        "response_target_tensor": torch.tensor(padded_dict["response_target_padded"], dtype=torch.long),
    }


train_tensor = to_tensor_dataset(train_padded)
val_tensor = to_tensor_dataset(val_padded)
test_tensor = to_tensor_dataset(test_padded)

print("\nHasil konversi ke torch.Tensor:")
for name, tensor_dict in [("Train", train_tensor), ("Val", val_tensor), ("Test", test_tensor)]:
    print(f"\n{name}:")
    for k, v in tensor_dict.items():
        print(f"  {k:25s}: shape={tuple(v.shape)}, dtype={v.dtype}")


Hasil konversi ke torch.Tensor:

Train:
  input_tensor             : shape=(1635, 70), dtype=torch.int64
  bspan_input_tensor       : shape=(1635, 15), dtype=torch.int64
  bspan_target_tensor      : shape=(1635, 15), dtype=torch.int64
  response_input_tensor    : shape=(1635, 50), dtype=torch.int64
  response_target_tensor   : shape=(1635, 50), dtype=torch.int64

Val:
  input_tensor             : shape=(553, 61), dtype=torch.int64
  bspan_input_tensor       : shape=(553, 15), dtype=torch.int64
  bspan_target_tensor      : shape=(553, 15), dtype=torch.int64
  response_input_tensor    : shape=(553, 42), dtype=torch.int64
  response_target_tensor   : shape=(553, 42), dtype=torch.int64

Test:
  input_tensor             : shape=(556, 72), dtype=torch.int64
  bspan_input_tensor       : shape=(556, 13), dtype=torch.int64
  bspan_target_tensor      : shape=(556, 13), dtype=torch.int64
  response_input_tensor    : shape=(556, 43), dtype=torch.int64
  response_target_tensor   : shape=(556, 43),